[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/62_moe_load_balance_solution.ipynb)

# 🟡 Solution: MoE Load Balancing Loss

Reference solution for the auxiliary load balancing loss from Switch Transformer / DeepSeek-style Mixture-of-Experts. Penalizes uneven token distribution across experts.

$$f_i = \frac{1}{N}\sum_t \mathbf{1}[\arg\max_e r_t = i], \quad P_i = \frac{1}{N}\sum_t \text{softmax}(r_t)_i$$

$$\mathcal{L}_{\text{aux}} = E \cdot \sum_{i=1}^{E} f_i \cdot P_i$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def moe_load_balance_loss(router_logits, num_experts):
    N = router_logits.shape[0]
    # f_i: fraction of tokens routed to each expert (argmax, non-differentiable)
    assignments = router_logits.argmax(dim=-1)  # (N,)
    f = torch.zeros(num_experts, device=router_logits.device)
    for e in range(num_experts):
        f[e] = (assignments == e).float().mean()
    # P_i: mean router probability per expert (differentiable via softmax)
    probs = torch.softmax(router_logits, dim=-1)  # (N, num_experts)
    P = probs.mean(dim=0)                          # (num_experts,)
    return num_experts * (f * P).sum()

In [ ]:
# Uniform routing gives loss = 1.0
# Identical logits -> uniform softmax -> uniform argmax -> f_i = 1/E, P_i = 1/E
# L_aux = E * E * (1/E)^2 = 1.0
num_experts = 4
N_tokens = 100
logits_uniform = torch.zeros(N_tokens, num_experts)
loss_uniform = moe_load_balance_loss(logits_uniform, num_experts)
print(f"Uniform routing loss: {loss_uniform.item():.4f}  (expected 1.0)")

# Collapsed routing gives loss > 1
logits_collapsed = torch.zeros(N_tokens, num_experts)
logits_collapsed[:, 0] = 100.0
loss_collapsed = moe_load_balance_loss(logits_collapsed, num_experts)
print(f"Collapsed routing loss: {loss_collapsed.item():.4f}  (expected > 1.0)")

# Gradient flows through router_logits
torch.manual_seed(0)
logits_grad = torch.randn(16, num_experts, requires_grad=True)
loss_grad = moe_load_balance_loss(logits_grad, num_experts)
loss_grad.backward()
print(f"Gradient exists: {logits_grad.grad is not None}")

In [ ]:
from torch_judge import check
check("moe_load_balance")